# Stream Media API Demo

This notebook demonstrates the full API surface of the **stream-media** project — a Rust microservices platform for managing and streaming media files.

## Architecture

| Service | Port | Description |
|---|---|---|
| **Gateway** | 3000 | Reverse proxy — single entry point for all API calls |
| **Catalog Service** | 3001 | Media metadata CRUD, SMB source management (SQLite) |
| **Streaming Service** | 3002 | File upload, HLS/DASH transcoding, adaptive streaming |
| **User Service** | 3003 | User management with password hashing (SQLite) |

## Streaming Methods

| Method | Description |
|---|---|
| **HLS** | Adaptive bitrate — videos transcoded to 360p/720p/1080p `.m3u8` + `.ts` segments |
| **DASH** | Adaptive bitrate — videos transcoded to `.mpd` manifest + `.m4s` segments |
| **HTTP Range** | Direct file serving with byte-range support, no transcoding |

## Prerequisites

```bash
# 1. Run initial setup (creates .env with admin credentials and streaming method)
./setup.sh

# 2. Start services
docker compose up -d
```

In [ ]:
import requests
import json
import time

BASE_URL = "http://localhost:3000"

def pp(resp):
    """Pretty-print an HTTP response."""
    print(f"{resp.request.method} {resp.request.url}")
    print(f"Status: {resp.status_code}")
    if resp.headers.get("content-type", "").startswith("application/json"):
        print(json.dumps(resp.json(), indent=2))
    elif resp.text:
        print(resp.text[:500])
    print()

---

## 1. User Management

Users have optional passwords (hashed with argon2) and an `is_admin` flag. The admin user is auto-seeded on startup from environment variables set by `setup.sh`.

### 1.1 Create Users

In [ ]:
# Create users — password is optional
users_to_create = [
    {
        "username": "jpareja",
        "email": "jpareja@example.com",
        "display_name": "Joel Pareja",
        "password": "secretpass123"
    },
    {
        "username": "asmith",
        "email": "asmith@example.com",
        "display_name": "Alice Smith",
        "password": "alicepass"
    },
    {
        "username": "bwilson",
        "email": "bwilson@example.com"
        # no display_name or password
    },
]

created_users = []
for user in users_to_create:
    resp = requests.post(f"{BASE_URL}/api/users", json=user)
    pp(resp)
    if resp.status_code == 201:
        created_users.append(resp.json())

### 1.2 List All Users

In [ ]:
# List all users (supports pagination with ?limit=&offset=)
resp = requests.get(f"{BASE_URL}/api/users")
pp(resp)

### 1.3 Search Users

Search across username, email, and display name.

In [ ]:
# Search users by keyword (matches username, email, or display_name)
resp = requests.get(f"{BASE_URL}/api/users", params={"search": "alice"})
pp(resp)

### 1.4 Get a Single User

In [ ]:
# Fetch a single user by ID
user_id = created_users[0]["id"]
resp = requests.get(f"{BASE_URL}/api/users/{user_id}")
pp(resp)

### 1.5 Update a User

Partial updates — only the fields you include will be changed.

In [ ]:
# Update user — only send fields you want to change
resp = requests.put(f"{BASE_URL}/api/users/{user_id}", json={
    "display_name": "Joel M. Pareja"
})
pp(resp)

### 1.6 Delete a User

In [ ]:
# Delete the third user
delete_id = created_users[2]["id"]
resp = requests.delete(f"{BASE_URL}/api/users/{delete_id}")
pp(resp)

# Verify deletion — should return 404
resp = requests.get(f"{BASE_URL}/api/users/{delete_id}")
pp(resp)

### 1.7 Uniqueness Constraints

Username and email must be unique. Duplicates return `400 Bad Request`.

In [ ]:
# Attempting to create a user with a duplicate username
resp = requests.post(f"{BASE_URL}/api/users", json={
    "username": "jpareja",          # already exists
    "email": "different@example.com"
})
pp(resp)  # 400 Bad Request

---

## 2. Media Catalog Management

The Catalog Service stores media metadata. Media items track their `transcode_status` (`pending`, `processing`, `ready`, `failed`, or `not_applicable`) and `transcode_format` (`hls`, `dash`, or `null`).

### 2.1 Create Media Entries (Metadata Only)

In [ ]:
# Create media entries (metadata only — no file upload)
media_items = [
    {
        "title": "Bohemian Rhapsody",
        "description": "Queen's iconic rock opera from A Night at the Opera (1975)",
        "media_type": "audio",
        "format": "flac",
        "duration_secs": 354.0
    },
    {
        "title": "Comfortably Numb",
        "description": "Pink Floyd classic from The Wall (1979)",
        "media_type": "audio",
        "format": "mp3",
        "duration_secs": 382.0
    },
    {
        "title": "Nature Documentary - Coral Reefs",
        "description": "4K underwater footage of the Great Barrier Reef",
        "media_type": "video",
        "format": "mp4",
        "duration_secs": 2700.0
    },
]

created_media = []
for item in media_items:
    resp = requests.post(f"{BASE_URL}/api/media", json=item)
    pp(resp)
    if resp.status_code == 201:
        created_media.append(resp.json())

### 2.2 List All Media

In [ ]:
# List all media items
resp = requests.get(f"{BASE_URL}/api/media")
pp(resp)

### 2.3 Filter by Media Type

In [ ]:
# Filter by media type: "audio" or "video"
resp = requests.get(f"{BASE_URL}/api/media", params={"media_type": "audio"})
pp(resp)

### 2.4 Search Media by Title

In [ ]:
# Search by title (case-insensitive substring match)
resp = requests.get(f"{BASE_URL}/api/media", params={"search": "bohemian"})
pp(resp)

### 2.5 Pagination

In [ ]:
# Paginate results: first page (1 item)
resp = requests.get(f"{BASE_URL}/api/media", params={"limit": 1, "offset": 0})
pp(resp)

# Second page
resp = requests.get(f"{BASE_URL}/api/media", params={"limit": 1, "offset": 1})
pp(resp)

### 2.6 Get a Single Media Item

In [ ]:
# Fetch a single media item by ID
media_id = created_media[0]["id"]
resp = requests.get(f"{BASE_URL}/api/media/{media_id}")
pp(resp)

### 2.7 Update Media Metadata

In [ ]:
# Partial update — only title and description
resp = requests.put(f"{BASE_URL}/api/media/{media_id}", json={
    "title": "Bohemian Rhapsody (Remastered 2011)",
    "description": "Remastered version from the 2011 deluxe edition"
})
pp(resp)

### 2.8 Delete a Media Item

In [ ]:
# Delete media item
delete_media_id = created_media[1]["id"]
resp = requests.delete(f"{BASE_URL}/api/media/{delete_media_id}")
pp(resp)

# Confirm it's gone
resp = requests.get(f"{BASE_URL}/api/media/{delete_media_id}")
pp(resp)

---

## 3. File Upload and Direct Streaming

The Streaming Service handles multipart file uploads and serves files with HTTP range support. When the streaming method is `hls` or `dash`, video uploads automatically trigger transcoding.

### 3.1 Upload a Media File

In [ ]:
# Create a sample file for upload demo
sample_content = b"RIFF" + b"\x00" * 1020  # minimal fake WAV header (1KB)
sample_path = "/tmp/demo-sample.wav"
with open(sample_path, "wb") as f:
    f.write(sample_content)

# Upload via multipart form
# NOTE: the "format" field must appear BEFORE the "file" field
resp = requests.post(f"{BASE_URL}/upload", files={
    "title":       (None, "Demo Upload Track"),
    "description": (None, "A sample file uploaded via the API"),
    "media_type":  (None, "audio"),
    "format":      (None, "wav"),
    "file":        ("demo-sample.wav", open(sample_path, "rb"), "audio/wav"),
})
pp(resp)

if resp.status_code == 201:
    uploaded_item = resp.json()
    uploaded_id = uploaded_item["id"]

### 3.2 Stream the Full File

A simple GET request downloads the entire file.

In [ ]:
# Stream the full file (HTTP 200)
resp = requests.get(f"{BASE_URL}/stream/{uploaded_id}")
print(f"GET {resp.url}")
print(f"Status: {resp.status_code}")
print(f"Content-Type: {resp.headers.get('content-type')}")
print(f"Content-Length: {resp.headers.get('content-length')}")
print(f"Accept-Ranges: {resp.headers.get('accept-ranges')}")
print(f"Body size: {len(resp.content)} bytes")
print()

### 3.3 HTTP Range Requests

Range requests return `206 Partial Content` with a `Content-Range` header — essential for seeking in audio/video players.

In [ ]:
# Range request — fetch bytes 0 through 99 (first 100 bytes)
resp = requests.get(f"{BASE_URL}/stream/{uploaded_id}", headers={"Range": "bytes=0-99"})
print(f"Status: {resp.status_code}")  # 206 Partial Content
print(f"Content-Range: {resp.headers.get('content-range')}")
print(f"Content-Length: {resp.headers.get('content-length')}")
print(f"Body size: {len(resp.content)} bytes")
print()

# Open-ended range — from byte 512 to end of file
resp = requests.get(f"{BASE_URL}/stream/{uploaded_id}", headers={"Range": "bytes=512-"})
print(f"Status: {resp.status_code}")
print(f"Content-Range: {resp.headers.get('content-range')}")
print(f"Content-Length: {resp.headers.get('content-length')}")
print(f"Body size: {len(resp.content)} bytes")
print()

# Suffix range — last 256 bytes
resp = requests.get(f"{BASE_URL}/stream/{uploaded_id}", headers={"Range": "bytes=-256"})
print(f"Status: {resp.status_code}")
print(f"Content-Range: {resp.headers.get('content-range')}")
print(f"Content-Length: {resp.headers.get('content-length')}")
print(f"Body size: {len(resp.content)} bytes")

---

## 4. SMB Source Management

Register Samba/CIFS network shares as media sources (Kodi-style). Sources must be mounted before media can be registered from them.

### 4.1 Register an SMB Source

In [ ]:
# Register a new SMB share — password is stored securely, never returned in responses
resp = requests.post(f"{BASE_URL}/api/sources/smb", json={
    "name": "NAS Media Share",
    "server": "nas.local",
    "share_name": "media",
    "username": "reader",
    "password": "secret"
})
pp(resp)

if resp.status_code == 201:
    smb_source = resp.json()
    source_id = smb_source["id"]

### 4.2 List SMB Sources

In [ ]:
# List all registered SMB sources (note: is_mounted reflects live mount status)
resp = requests.get(f"{BASE_URL}/api/sources/smb")
pp(resp)

### 4.3 Mount / Unmount a Source

Mounting runs `mount -t cifs` on the server. The share must be accessible from the host.
Credentials are passed via a temporary file (never visible in `/proc`).

In [ ]:
# Mount the share (requires the SMB server to be reachable)
# resp = requests.post(f"{BASE_URL}/api/sources/smb/{source_id}/mount")
# pp(resp)

# Unmount when done
# resp = requests.post(f"{BASE_URL}/api/sources/smb/{source_id}/unmount")
# pp(resp)

print("(Skipped — requires a reachable SMB server on the network)")

### 4.4 Register Media from a Mounted Source

Once a source is mounted, register media files by referencing the `source_id` and path within the share. The file must exist on the mounted filesystem — the system validates this and reads the file size automatically.

In [ ]:
# Register media from a mounted SMB source
# (Requires the source to be mounted — see section 4.3)
#
# resp = requests.post(f"{BASE_URL}/api/media/register-smb", json={
#     "title": "Network Movie",
#     "media_type": "video",
#     "format": "mkv",
#     "source_id": source_id,
#     "path": "movies/example.mkv"
# })
# pp(resp)

print("(Skipped — requires a mounted SMB source with files)")

---

## 5. Transcoding and Adaptive Streaming

When the streaming method is `hls` or `dash`, video uploads automatically trigger transcoding into three quality variants (360p, 720p, 1080p). Transcoding can also be triggered manually.

### 5.1 Trigger Transcoding Manually

In [ ]:
# Trigger transcoding for a video item
# (Requires a real video file — the fake WAV from section 3 won't transcode)
# The format (HLS or DASH) is determined by the STREAMING_METHOD env var.
#
# resp = requests.post(f"{BASE_URL}/transcode/{video_media_id}")
# pp(resp)  # 202 Accepted with { media_id, transcode_status, transcode_format }

print("Example response:")
print(json.dumps({
    "media_id": "550e8400-e29b-41d4-a716-446655440000",
    "transcode_status": "pending",
    "transcode_format": "hls"
}, indent=2))

### 5.2 Check Transcode Status

Poll the status endpoint to know when transcoding is complete.

In [ ]:
# Check transcode status — returns status + available variants when ready
#
# resp = requests.get(f"{BASE_URL}/transcode/{video_media_id}/status")
# pp(resp)

print("Example response (when complete):")
print(json.dumps({
    "media_id": "550e8400-e29b-41d4-a716-446655440000",
    "transcode_status": "ready",
    "transcode_format": "hls",
    "variants": ["360p", "720p", "1080p"]
}, indent=2))

### 5.3 HLS Streaming

When transcoding is complete with format `hls`, adaptive streams are available at these endpoints. Players like `hls.js` or Safari's native `<video>` tag consume the master playlist and switch quality automatically.

In [ ]:
# Fetch the HLS master playlist (lists all quality variants)
# resp = requests.get(f"{BASE_URL}/stream/{video_media_id}/hls/master.m3u8")
# print(resp.text)

print("Example master playlist (master.m3u8):")
print("""#EXTM3U
#EXT-X-STREAM-INF:BANDWIDTH=896000,RESOLUTION=640x360
360p/playlist.m3u8
#EXT-X-STREAM-INF:BANDWIDTH=2628000,RESOLUTION=1280x720
720p/playlist.m3u8
#EXT-X-STREAM-INF:BANDWIDTH=5192000,RESOLUTION=1920x1080
1080p/playlist.m3u8""")
print()

# Fetch a variant playlist
# resp = requests.get(f"{BASE_URL}/stream/{video_media_id}/hls/720p/playlist.m3u8")
#
# Fetch a segment
# resp = requests.get(f"{BASE_URL}/stream/{video_media_id}/hls/720p/segment_000.ts")

### 5.4 DASH Streaming

When transcoding is complete with format `dash`, MPEG-DASH streams are available. Players like `dash.js` or `Shaka Player` consume the MPD manifest.

In [ ]:
# Fetch the DASH manifest
# resp = requests.get(f"{BASE_URL}/stream/{video_media_id}/dash/manifest.mpd")
# print(resp.text)  # XML MPD manifest
#
# Fetch an init segment (representation IDs are numeric: 0, 1, 2, ...)
# resp = requests.get(f"{BASE_URL}/stream/{video_media_id}/dash/0/init.m4s")
#
# Fetch a media chunk
# resp = requests.get(f"{BASE_URL}/stream/{video_media_id}/dash/0/chunk-00001.m4s")

print("DASH endpoints:")
print(f"  Manifest:  GET /stream/{{id}}/dash/manifest.mpd")
print(f"  Init seg:  GET /stream/{{id}}/dash/{{repr}}/init.m4s")
print(f"  Chunk:     GET /stream/{{id}}/dash/{{repr}}/chunk-00001.m4s")

---

## 6. API Reference

### Endpoints Summary

| Method | Path | Description |
|---|---|---|
| **Users** | | |
| `POST` | `/api/users` | Create user (optional `password`) |
| `GET` | `/api/users` | List users (`search`, `limit`, `offset`) |
| `GET` | `/api/users/{id}` | Get user |
| `PUT` | `/api/users/{id}` | Update user (partial) |
| `DELETE` | `/api/users/{id}` | Delete user |
| **Media** | | |
| `POST` | `/api/media` | Create media metadata |
| `GET` | `/api/media` | List media (`search`, `media_type`, `limit`, `offset`) |
| `GET` | `/api/media/{id}` | Get media |
| `PUT` | `/api/media/{id}` | Update media (partial) |
| `DELETE` | `/api/media/{id}` | Delete media |
| `POST` | `/api/media/register-smb` | Register media from mounted SMB source |
| `PATCH` | `/api/media/{id}/transcode-status` | Update transcode status (internal) |
| **SMB Sources** | | |
| `POST` | `/api/sources/smb` | Register SMB share |
| `GET` | `/api/sources/smb` | List sources |
| `GET` | `/api/sources/smb/{id}` | Get source |
| `PUT` | `/api/sources/smb/{id}` | Update source |
| `DELETE` | `/api/sources/smb/{id}` | Delete source (fails if media references it) |
| `POST` | `/api/sources/smb/{id}/mount` | Mount the share |
| `POST` | `/api/sources/smb/{id}/unmount` | Unmount the share |
| **Upload & Direct Streaming** | | |
| `POST` | `/upload` | Upload media file (multipart) |
| `GET` | `/stream/{id}` | Stream original file (supports Range) |
| **HLS** | | |
| `GET` | `/stream/{id}/hls/master.m3u8` | Master playlist |
| `GET` | `/stream/{id}/hls/{variant}/playlist.m3u8` | Variant playlist (360p/720p/1080p) |
| `GET` | `/stream/{id}/hls/{variant}/{segment}` | TS segment |
| **DASH** | | |
| `GET` | `/stream/{id}/dash/manifest.mpd` | MPD manifest |
| `GET` | `/stream/{id}/dash/{repr}/{file}` | Init/chunk segments (.m4s/.mp4) |
| **Transcode** | | |
| `POST` | `/transcode/{id}` | Trigger transcoding (format per STREAMING_METHOD) |
| `GET` | `/transcode/{id}/status` | Transcode job status + available variants |

### Environment Variables

| Variable | Default | Description |
|---|---|---|
| `ADMIN_USERNAME` | — | Admin user seeded on startup |
| `ADMIN_EMAIL` | — | Admin email |
| `ADMIN_PASSWORD` | — | Admin password (hashed with argon2) |
| `STREAMING_METHOD` | `hls` | `hls`, `dash`, or `range` |
| `TRANSCODE_MAX_JOBS` | `2` | Max concurrent ffmpeg processes |
| `SMB_MOUNT_BASE` | `/mnt/smb` | Base path for SMB mounts |
| `MEDIA_STORE_PATH` | `./media-store` | Where uploaded files and transcoded output live |

---

## 7. Cleanup

Remove the demo data created during this notebook run.

In [ ]:
# Delete all users created in this session
for user in created_users:
    resp = requests.delete(f"{BASE_URL}/api/users/{user['id']}")
    status = "deleted" if resp.status_code == 204 else f"skipped ({resp.status_code})"
    print(f"User {user.get('username', user['id'])}: {status}")

# Delete all media created in this session
for item in created_media:
    resp = requests.delete(f"{BASE_URL}/api/media/{item['id']}")
    status = "deleted" if resp.status_code == 204 else f"skipped ({resp.status_code})"
    print(f"Media {item.get('title', item['id'])}: {status}")

# Delete uploaded item
if 'uploaded_id' in dir():
    resp = requests.delete(f"{BASE_URL}/api/media/{uploaded_id}")
    status = "deleted" if resp.status_code == 204 else f"skipped ({resp.status_code})"
    print(f"Uploaded item: {status}")

# Delete SMB source
if 'source_id' in dir():
    resp = requests.delete(f"{BASE_URL}/api/sources/smb/{source_id}")
    status = "deleted" if resp.status_code == 204 else f"skipped ({resp.status_code})"
    print(f"SMB source: {status}")

print("\nCleanup complete.")